# KAT-VAD v2 — P6 / **pilot batch 1: A0 + A1** (seed 2099)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§4** (pilot read-out), **§13 D16** (two batches), **§14 O1–O7**
(how the read-out and D5 are computed). Batch 2 (A2/A3) waits for E2(d) (`p4_e2d.ipynb`).

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight; stage T2 + CLIP caches to VM disk | no |
| 2 | **v2 dataset dir** (`v2_dataset`): train = T2-train **minus T2-val**, eval file = T2-val windows; KNN cache rebuilt on it | no |
| 3 | **bake A1** (`build_v2_inputs fit --crn R2`), + `apply --stride 3` on `dota_dev` (O5 only) | no |
| 4 | **train A0 and A1**, phase 4's recipe, seed 2099, chunked + resumable, verified on Drive (J10) | **yes** |
| 5 | **diagnostics** (O1–O5) for A0, A1 and the three phase-4 references → **D5** (O6) | yes (or slow CPU) |
| 6 | record → stop | — |

**DoTA-dev is not printed (D4).** It is read unlabelled for the source-shortcut probe only (O5).
Everything new is written under `Thesis-V2/`; `Thesis/` is only read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/, v1 outputs (phase-4 references)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'
V1_OUTPUTS = Path(DRIVE) / 'outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_D5_MARGIN'), 'this upload predates addendum §14 -- re-upload branch v2'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # 3
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT          # 5
NUM_EPOCHS, CHUNK_EPOCHS = 20, 5                         # phase 4's recipe (O7)
SEED       = 2099                                        # D4: outside 2024-2028
CRN        = 'R2'                                        # D11

T2        = constants.DATA_ROOT / DATASET
CLIP_DIR  = constants.CLIP_CACHE_DIR / DATASET
DOTA      = constants.DATA_ROOT / 'DoTA'
LABELS    = DOTA / 'labels_s8'                            # defs.json only: no DoTA label is read
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_DATA   = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'
V2_CACHE  = Path(DRIVE_V2) / 'cache'
A1_T2     = V2_CACHE / 'v2' / f'A1_{CRN}' / DATASET
A1_DOTA   = V2_CACHE / 'v2' / f'A1_{CRN}' / 'DoTA_s3_from_s1'
KNN_V2    = V2_CACHE / 'knn' / f'{DATASET}_v2' / constants.KNN_CACHE_FILENAME
PILOT     = constants.OUTPUT_ROOT / 'v2_pilot'           # <arm>/s2099/{run files, diag/}
REFS = {                                                 # D5 references (O6): finished phase-4 KIP-off arms
    's2024': V1_OUTPUTS / 'DADA2000_orig_phase4' / 's2024' / 'stage2_kip_off' / 'checkpoint_last.pt',
    's2025': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2025' / 'stage2_kip_off' / 'checkpoint_last.pt',
    's2026': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2026' / 'stage2_kip_off' / 'checkpoint_last.pt',
}
P4_CONFIG = V1_OUTPUTS / 'DADA2000_orig_phase4' / 's2024' / 'stage2_kip_off' / 'config.yaml'
LOCAL = Path('/content/pilot')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('T2', 'CLIP_DIR', 'DOTA_S1', 'V2_DATA', 'A1_T2', 'KNN_V2', 'PILOT'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

In [ ]:
import hashlib
import json
import shutil
import subprocess
import time

import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst):
    """Copy a Drive dir/file to VM disk once (bit-identical; C2 untouched)."""
    if dst.exists():
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight + stage

In [ ]:
from core.data.v2_splits import load_split

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 14. P6 pilot implementation choices' in addendum and 'D16' in addendum, 'upload predates §13-§14'
dev = load_split(constants.V2_SPLIT_DOTA_DEV)
val = load_split(constants.V2_SPLIT_T2_VAL)
assert len(dev) == 702 and len(val) == 219, (len(dev), len(val))
for path in (T2, CLIP_DIR, DOTA_S1, LABELS / constants.DEFS_FILENAME, P4_CONFIG, *REFS.values()):
    assert path.exists(), f'missing {path}'

L_T2 = stage(T2, LOCAL / 'stage' / 'corpus')
L_CLIP = stage(CLIP_DIR, LOCAL / 'stage' / 'clip')
L_DOTA_S1 = LOCAL / 'stage' / 'DoTA_s1_ncc_dev'
L_DOTA_S1.mkdir(parents=True, exist_ok=True)
for v in dev:
    if not (L_DOTA_S1 / f'{v}.npy').exists():
        shutil.copy2(DOTA_S1 / f'{v}.npy', L_DOTA_S1 / f'{v}.npy')
print(f'T2-val {len(val)} sources | DoTA-dev {len(dev)} clips staged (unlabelled use only)')

## 2. v2 dataset dir (proposal §7.2) + KNN cache

`v2_dataset` refuses to write unless its label arithmetic reproduces **every** T2-test window's labels of the parent
dir. The KNN cache is keyed to the train split, so it is rebuilt here on raw CLIP (shared by A0 and A1, so DVS splices
the same neighbours in both arms).

In [ ]:
from core.tools import v2_dataset

L_V2 = LOCAL / 'stage' / 'corpus_v2'
if not (L_V2 / v2_dataset.MANIFEST_FILENAME).exists():
    run([sys.executable, '-m', 'core.tools.v2_dataset', '--data-dir', L_T2, '--out-dir', L_V2])
manifest = json.loads((L_V2 / v2_dataset.MANIFEST_FILENAME).read_text())
print(json.dumps({k: manifest[k] for k in ('train_sources', 'train_windows', 't2_val_sources',
                                           't2_val_windows', 't2_val_windows_abnormal',
                                           'label_arithmetic_checked_on_parent_test_windows')}, indent=1))
assert manifest['t2_val_sources'] == 219

L_KNN = LOCAL / 'stage' / 'knn_v2.npz'
if not L_KNN.exists():
    run([sys.executable, '-m', 'core.data.knn_cache', '--data-dir', L_V2, '--dataset', DATASET,
         '--clip-dir', L_CLIP, '--output', L_KNN])
import numpy as np  # noqa: E402
labels = json.loads((L_V2 / constants.LABELS_TRAIN_FILENAME).read_text())
with np.load(L_KNN, allow_pickle=True) as z:
    anomalies, pool = set(z['anomaly_ids'].tolist()), set(z['neighbor_ids'].ravel().tolist())
assert anomalies == {i for i, y in labels.items() if y == 1}, 'KNN keys != v2 train abnormal windows'
assert pool <= {i for i, y in labels.items() if y == 0}, 'KNN pool reaches outside v2 train normals'
publish(L_V2, V2_DATA)
publish(L_KNN, KNN_V2)
print('v2 dataset dir + KNN cache published to Drive')

## 3. Bake A1 (`v2.crn=R2`, no motion)

`fit`: `s` on T2-train minus T2-val (the manifest records `train_ids_sha1`), every T2 source baked with its own R2
reference. `apply --stride 3`: DoTA-dev clips, each its own reference over its stride-3 rows (O5 only here).

In [ ]:
L_A1_T2 = LOCAL / 'stage' / 'A1_T2'
L_A1_DOTA = LOCAL / 'stage' / 'A1_DOTA'
if not (L_A1_T2 / constants.V2_INPUT_MANIFEST_FILENAME).exists():
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit', '--t2-dir', L_V2,
         '--clip-dir', L_CLIP, '--crn', CRN, '--motion', 'none', '--out-dir', L_A1_T2])
if not (L_A1_DOTA / constants.V2_INPUT_MANIFEST_FILENAME).exists():
    ids_file = LOCAL / 'dota_dev.txt'
    ids_file.write_text(''.join(f'{v}\n' for v in dev))
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', L_A1_T2,
         '--clip-dir', L_DOTA_S1, '--stride', constants.V2_E1_STRIDE_BC, '--ids-file', ids_file,
         '--out-dir', L_A1_DOTA])
m = json.loads((L_A1_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
print(f"A1: crn {m['crn']} s={m['s']:.4f} on {m['train_sources']} T2-train sources (val {m['t2_val_sources']}); "
      f"baked {m['baked_ids']} sources")
assert m['crn'] == CRN and m['motion'] == 'none' and m['t2_val_sources'] == 219
publish(L_A1_T2, A1_T2)
publish(L_A1_DOTA, A1_DOTA)

## 4. Train A0 and A1 — chunked, resumable, verified

5 epochs per call, `--resume` in between, synced to Drive after every chunk. The skip/resume decision reads the
**checkpoint's own `global_step`** (Amendment 3). **Re-run this cell after a disconnect.**

In [ ]:
ARMS = {
    'A0': {'clip': L_CLIP, 'extra': []},
    'A1': {'clip': L_A1_T2, 'extra': ['--set', f'v2.crn={CRN}']},
}


def train_cmd(arm, out_dir):
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
            '--set', f'train.num_epochs={NUM_EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={KERNEL}', '--set', f'loss.mil_topk_pct={TOPK_PCT}',
            '--set', 'kip.enabled=false', *ARMS[arm]['extra'],
            '--data-dir', L_V2, '--clip-dir', ARMS[arm]['clip'], '--knn-cache', L_KNN,
            '--output-dir', out_dir]


def sync(local, drive):
    drive.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            publish(local / name, drive / name)


def final_step(local):
    return metrics_last(local / 'metrics.jsonl')['global_step']


for arm in ARMS:
    drive_dir, local = PILOT / arm / f's{SEED}', LOCAL / 'runs' / arm
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if last and last['epoch'] + 1 == NUM_EPOCHS and ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        print(f'{arm}: finished + verified on Drive (step {last["global_step"]}) -- skipping')
        continue
    local.mkdir(parents=True, exist_ok=True)
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    while True:
        done = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done >= NUM_EPOCHS:
            break
        cmd = train_cmd(arm, local)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== {arm}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd, log=LOCAL / f'{arm}_train.log', quiet_steps=True)
        sync(local, drive_dir)
    assert ckpt_step(local / 'checkpoint_last.pt') == final_step(local), f'{arm}: J10 fails locally'
    sync(local, drive_dir)
    print(f'{arm}: trained to step {final_step(local)}, synced')

# Config check: A0 == phase 4 except the seed (and v2-only keys at "off"); A1 == A0 except v2.crn.
ref = yaml.safe_load(P4_CONFIG.read_text())
cfg = {arm: yaml.safe_load((PILOT / arm / f's{SEED}' / 'config.yaml').read_text()) for arm in ARMS}


def diff(a, b):
    out = set()
    for section in set(a) | set(b):
        sa, sb = a.get(section) or {}, b.get(section) or {}
        out |= {f'{section}.{k}' for k in set(sa) | set(sb) if sa.get(k) != sb.get(k)}
    return out


v2_only = {'model.motion_dim', 'v2.crn', 'v2.motion'}
d_a0 = diff(ref, cfg['A0']) - {'train.seed'}
assert all(k in v2_only for k in d_a0), f'A0 differs from phase 4 beyond the seed: {sorted(d_a0)}'
assert diff(cfg['A0'], cfg['A1']) == {'v2.crn'}, f'A1 differs from A0 beyond v2.crn: {diff(cfg["A0"], cfg["A1"])}'
for arm in ARMS:
    drive_dir = PILOT / arm / f's{SEED}'
    step, last = ckpt_step(drive_dir / 'checkpoint_last.pt'), metrics_last(drive_dir / 'metrics.jsonl')
    assert step == last['global_step'] and last['epoch'] + 1 == NUM_EPOCHS, f'{arm}: J10 FAILS on Drive'
    print(f'{arm}: Drive checkpoint step {step} == metrics last step (J10)')
print('configs: A0 == phase 4 but the seed; A1 == A0 but v2.crn')

## 5. Diagnostics (O1–O5) and **D5** (O6)

One forward pass per item. A0 first (A1's guardrail and position comparison read A0's `diag.json`), then A1, then the
three phase-4 references on A0's inputs for D5. DoTA-dev is used unlabelled (source-shortcut only).

In [ ]:
from core.tools import v2_diagnostics

L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME, LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
INPUTS = {'A0': (L_CLIP, L_DOTA_S1), 'A1': (L_A1_T2, L_A1_DOTA)}


def diagnose(name, ckpt, arm_inputs, out_dir, a0_diag=None):
    if (out_dir / v2_diagnostics.DIAG_JSON).exists():
        print(f'{name}: reusing {out_dir / v2_diagnostics.DIAG_JSON}')
    else:
        t2_in, dota_in = arm_inputs
        run([sys.executable, '-m', 'core.tools.v2_diagnostics', '--run', name, ckpt,
             '--data-dir', L_V2, '--t2-input-dir', t2_in, '--dota-input-dir', dota_in,
             '--dota-s1-dir', L_DOTA_S1, '--dota-data-dir', L_DOTA_LABELS,
             *(['--a0-diag', a0_diag] if a0_diag else []), '--out-dir', out_dir, '--device', 'auto'],
            log=LOCAL / f'{name}_diag.log')
    return json.loads((out_dir / v2_diagnostics.DIAG_JSON).read_text())


diags = {}
for arm in ('A0', 'A1'):
    out = PILOT / arm / f's{SEED}' / 'diag'
    a0_json = PILOT / 'A0' / f's{SEED}' / 'diag' / v2_diagnostics.DIAG_JSON
    diags[arm] = diagnose(f'{arm}_s{SEED}', PILOT / arm / f's{SEED}' / 'checkpoint_last.pt', INPUTS[arm], out,
                          a0_json if arm != 'A0' else None)
    print((out / v2_diagnostics.DIAG_MD).read_text())

refs = [diagnose(f'p4_{name}', ckpt, INPUTS['A0'], PILOT / 'd5_refs' / name) for name, ckpt in REFS.items()]
d5 = v2_diagnostics.d5(diags['A0'], refs)
(PILOT / 'd5.json').write_text(json.dumps(d5, indent=2))
print(json.dumps(d5, indent=2))
print('\nNOTE (O6): the phase-4 references trained on T2-val windows -- in-sample for them; expected higher.')

In [ ]:
rows = []
for arm, r in diags.items():
    g = r['guardrails']
    rows.append(f"| {arm} | {g['micro']:.4f} | {g['macro']:.4f} | {g['clip_oracle_micro']:.4f} | "
                f"{g['window_level_auc']:.4f} | {'PASS' if g['pass'] else 'FAIL'} | "
                f"{r['position_r2']['v_t']:.4f} | {r['source_shortcut_auc']['v_t']:.4f} | "
                f"{r['source_shortcut_auc']['y_bin']:.4f} |")
table = '\n'.join(['| arm | T2-val micro | macro | clip oracle | window AUC | guardrails | pos R² V^t | '
                   'shortcut V^t | shortcut y^bin |', '|---|---|---|---|---|---|---|---|---|', *rows])
text = (f'# v2 pilot batch 1 (seed {SEED}) — T2-val only (D4)\n\n{table}\n\n'
        f"D5 (O6): {'PASS' if d5['pass'] else 'FAIL'} -- micro Δ {d5['micro']['delta']:+.4f}, "
        f"macro Δ {d5['macro']['delta']:+.4f} vs the phase-4 mean (in-sample for them)\n")
(PILOT / 'batch1_readout.md').write_text(text)
print(text)

## 6. Record — then **stop**

Paste `v2_pilot/batch1_readout.md` and both `diag.md` back to Claude. A guardrail FAIL or a D5 FAIL blocks P7
(§5) until explained; neither is fixed by changing `lr`, `c` or the recipe (§4). Batch 2 (A2/A3) follows E2(d).